# Chapter 13. Coordinate and synchronization audit

**Learning objective.** Reconstruct native, head-corrected, anatomically rotated, and time-corrected layers while diagnosing reference drift, dropped frames, clock offset, and clock drift.

**Estimated time:** 90 minutes. **Exercises:** 13.2 and 13.3. Every coordinate and timing event is an explicitly **synthetic teaching fixture**. Passing the audit does not validate a physical EMA or audio system.

## Input and output contract

Inputs are session metadata, a stream manifest, immutable native coordinates and timestamps, recorded transformation matrices, and common synchronization events. Outputs preserve native values while adding derived coordinate layers, stream-specific quality decisions, two synchronization models, event-level residuals, diagnostics, and provenance.

In [ ]:
from pathlib import Path
import sys

start = Path.cwd().resolve()
for candidate in (start, *start.parents):
    if (candidate / "companion" / "data" / "MANIFEST.tsv").is_file():
        REPOSITORY_ROOT = candidate
        break
else:
    raise RuntimeError("Run from the repository root or one of its subdirectories.")
SOURCE_ROOT = REPOSITORY_ROOT / "companion" / "src"
if str(SOURCE_ROOT) not in sys.path:
    sys.path.insert(0, str(SOURCE_ROOT))
DATA_ROOT = REPOSITORY_ROOT / "companion" / "data" / "ch13" / "synthetic_multimodal"
if not DATA_ROOT.exists():
    raise FileNotFoundError("Chapter 13 fixture absent. Run companion/scripts/generate_ch13_fixture.py explicitly; no data are substituted silently.")
print("DATA STATUS: SYNTHETIC_TEACHING_FIXTURE — generated geometry, clocks, and trajectories, not human articulatory data.")

In [ ]:
from phonetic_research_companion.ch13 import run_coordinate_and_sync_audit
OUTPUT = REPOSITORY_ROOT / "companion" / "outputs" / "ch13" / "01_coordinate_and_sync_audit"
result = run_coordinate_and_sync_audit(DATA_ROOT / "multimodal_session.yaml", DATA_ROOT / "stream_manifest.tsv", DATA_ROOT / "native_coordinates.tsv", DATA_ROOT / "transformation_matrices.tsv", DATA_ROOT / "calibration_sync_events.tsv", OUTPUT)
assert result["native_unchanged"]
assert result["decisions"]["missing_frames"] == [80, 81]
assert result["decisions"]["drift_frames"]
for row in result["quality"]:
    print(row)
for row in result["sync_models"]:
    print(row)

## Learner gate

Locate the interval in which rigid reference geometry fails without consulting a condition label. Show how the failure moves the nominally stationary target after head correction. Compare the residuals from a one-offset correction with the fitted offset-and-drift model, retain the missing anchor and dropped frames, and state which spatial and temporal claims survive.

In [ ]:
required = ["stream_quality.tsv", "coordinate_transforms.tsv", "sync_model.tsv", "sync_residuals.tsv", "coordinate_sync_diagnostics.html", "run_provenance.json"]
assert all((OUTPUT / name).exists() for name in required)
print(result["decisions"])